In [1]:
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel,
)

In [2]:
# ==========================================
# Your existing data
# X 6D (30,6)
# y 1D (30,)
# ==========================================

# Example
X = np.load('function_7/initial_inputs.npy')
y = np.load('function_7/initial_outputs.npy')

X = np.vstack([X, [0.136137, 0.687594, 0.238254, 0.182404, 0.465656, 0.733703],
               [0.061062, 0.399794, 0.566951, 0.219760, 0.530846, 0.745732],
               [0.057896, 0.112680, 0.251070, 0.085255, 0.928944, 0.748098],
               [0.057896, 0.536970, 0.439210, 0.251539, 0.496588, 0.742380],
               [0.057896, 0.400647, 0.520291, 0.152111, 0.419030, 0.737726],
               [0.057896, 0.314520, 0.839546, 0.073659, 0.368571, 0.817120],
               [0.057896, 0.381146, 0.222571, 0.073659, 0.398422, 0.680939],
               [0.057896, 0.017979, 0.730104, 0.073659, 0.349826, 0.951014],
               [0.057896, 0.075797, 0.705507, 0.073659, 0.394764, 0.735312],
               [0.057896, 0.313038, 0.625712, 0.073659, 0.402919, 0.739475],
               [0.057896, 0.014256, 0.623715, 0.088011, 0.377643, 0.699568]])
y = np.append(y, [0.6127494075011818, 
                  1.2928053709159413,
                  0.15287457888484313,
                  1.0815078591777765,
                  1.8242353051630313,
                  1.2910519337152369,
                  1.2679294139972004,
                  0.8885485836283754,
                  1.5850934376403436,
                  1.7115368542475984,
                  1.9066125688793496])

In [3]:
# ==========================================
# Fit Gaussian Process
# ==========================================

kernel = (
    ConstantKernel(1.0)
    * Matern(length_scale=np.ones(X.shape[1]), nu=2.5)
    + WhiteKernel(noise_level=1e-6)
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=10,
    random_state=0
)

gp.fit(X, y)

# ==========================================
# Expected Improvement
# ==========================================

def expected_improvement(x, gp, y_best, xi=0.01):
    x = np.array(x).reshape(1, -1)

    mu, sigma = gp.predict(x, return_std=True)

    sigma = sigma[0]
    mu = mu[0]

    if sigma < 1e-12:
        return 0.0

    improvement = mu - y_best - xi
    Z = improvement / sigma

    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)

    return ei



# ==========================================
# Optimise acquisition function
# ==========================================

bounds = list(zip(X.min(axis=0), X.max(axis=0))) # replace with your limits

y_best = np.max(y)


def objective(x):
    return -expected_improvement(x, gp, y_best)


best_x = None
best_ei = -np.inf

for _ in range(5000):

    x0 = np.random.uniform(
        [b[0] for b in bounds],
        [b[1] for b in bounds]
    )

    res = minimize(
        objective,
        x0=x0,
        bounds=bounds,
        method="L-BFGS-B"
    )

    ei = -res.fun

    if ei > best_ei:
        best_ei = ei
        best_x = res.x

print("Next point to evaluate:")
next_point = np.round(best_x, decimals=6)
print(next_point)

print("Expected Improvement:")
print(best_ei)

C:\Users\mho2\AppData\Roaming\Python\Python310\site-packages\sklearn\gaussian_process\kernels.py:452: ConvergenceWarning: The optimal value found for dimension 3 of parameter k1__k2__length_scale is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


Next point to evaluate:
[0.057896 0.133275 0.009109 0.51697  0.380811 0.718914]
Expected Improvement:
0.008758609463151549
